In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text,engine
from dotenv import load_dotenv
import os 
load_dotenv()

# Configuration de la connexion PostgreSQL
# Remplacez les identifiants par les vôtres
DB_USER = os.getenv("DB_USER")
DB_PASSWORD =os.getenv("DB_PASSWORD")
DB_HOST =os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
db_url=f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(db_url)

# Chargement du fichier CSV
df=pd.read_csv(r"C:\Users\salli\OneDrive\Desktop\breifs 5\data\financecore_clean.csv")
df

In [ ]:
with engine.connect() as conn:
    # Suppression des tables si elles existent (pour réinitialisation)
    conn.execute(text("DROP TABLE IF EXISTS transactions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS agences CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS produits CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS segments CASCADE;"))

    # Table Segments
    conn.execute(text("""
        CREATE TABLE segments (
            id_segment SERIAL PRIMARY KEY,
            nom_segment VARCHAR(50) UNIQUE NOT NULL
        );
    """))

    # Table Agences
    conn.execute(text("""
        CREATE TABLE agences (
            id_agence SERIAL PRIMARY KEY,
            nom_agence VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    # Table Produits
    conn.execute(text("""
        CREATE TABLE produits (
            id_produit SERIAL PRIMARY KEY,
            nom_produit VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    # Table Clients
    conn.execute(text("""
        CREATE TABLE clients (
            client_id VARCHAR(50) PRIMARY KEY,
            score_credit INT,
            id_segment INT REFERENCES segments(id_segment)
        );
    """))

    # Table Transactions
    conn.execute(text("""
        CREATE TABLE transactions (
            transaction_id VARCHAR(50) PRIMARY KEY,
            date_transaction TIMESTAMP,
            montant_eur DECIMAL(15, 2),
            type_operation VARCHAR(50),
            statut VARCHAR(20),
            client_id VARCHAR(50) REFERENCES clients(client_id),
            id_agence INT REFERENCES agences(id_agence),
            id_produit INT REFERENCES produits(id_produit)
        );
    """))
    conn.commit()

In [ ]:
# 1. Insertion des tables de référence (Dimensions) 
def insert_dim(table_name, column_csv, column_db):
    # القيم unique من CSV
    unique_values = df[column_csv].dropna().unique()

    # القيم الموجودة في DB
    existing = pd.read_sql(f"SELECT {column_db} FROM {table_name}", engine)
    existing_values = set(existing[column_db])

    # غير القيم الجديدة فقط
    new_values = [v for v in unique_values if v not in existing_values]

    if new_values:
        dim_df = pd.DataFrame({column_db: new_values})
        dim_df.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"{len(new_values)} nouvelles valeurs ajoutées dans {table_name}")
    else:
        print(f"Aucune nouvelle valeur à insérer dans {table_name}")


# Insert dimensions
insert_dim('segments', 'segment_client', 'nom_segment')
insert_dim('agences', 'agence', 'nom_agence')
insert_dim('produits', 'produit', 'nom_produit')

# 2. Récupération des mappings IDs
segments_map = pd.read_sql(
    "SELECT id_segment, nom_segment FROM segments", engine
).set_index('nom_segment')['id_segment'].to_dict()

agences_map = pd.read_sql(
    "SELECT id_agence, nom_agence FROM agences", engine
).set_index('nom_agence')['id_agence'].to_dict()

produits_map = pd.read_sql(
    "SELECT id_produit, nom_produit FROM produits", engine
).set_index('nom_produit')['id_produit'].to_dict()


# 3. Insertion des Clients
clients_df = df[['client_id', 'score_credit_client', 'segment_client']].copy()

# حذف التكرار حسب client_id
clients_df = clients_df.drop_duplicates(subset=['client_id'], keep='first')

# mapping
clients_df['id_segment'] = clients_df['segment_client'].map(segments_map)

clients_final = clients_df[['client_id', 'score_credit_client', 'id_segment']].rename(
    columns={'score_credit_client': 'score_credit'}
)

# حذف clients لي راهوم فـ DB
existing_clients = pd.read_sql("SELECT client_id FROM clients", engine)
existing_ids = set(existing_clients['client_id'])

clients_final = clients_final[~clients_final['client_id'].isin(existing_ids)]

# insertion
if not clients_final.empty:
    clients_final.to_sql('clients', engine, if_exists='append', index=False)
    print(f"{len(clients_final)} clients insérés avec succès.")
else:
    print("Aucun nouveau client à insérer.")


# 4. Insertion des Transactions
tx_df = df.copy()

tx_df['id_agence'] = tx_df['agence'].map(agences_map)
tx_df['id_produit'] = tx_df['produit'].map(produits_map)

tx_final = tx_df[['transaction_id', 'date_transaction', 'montant_eur',
                  'type_operation', 'statut', 'client_id',
                  'id_agence', 'id_produit']]

# حذف التكرار
tx_final = tx_final.drop_duplicates(subset=['transaction_id'])

# حذف transactions لي راهوم فـ DB
existing_tx = pd.read_sql("SELECT transaction_id FROM transactions", engine)
existing_tx_ids = set(existing_tx['transaction_id'])

tx_final = tx_final[~tx_final['transaction_id'].isin(existing_tx_ids)]

# insertion
if not tx_final.empty:
    tx_final.to_sql('transactions', engine, if_exists='append', index=False)
    print("Chargement des transactions terminé avec succès.")
else:
    print("Aucune nouvelle transaction à insérer.")

# 4

In [ ]:
# Requête : Moyenne des transactions par agence et par mois
query_kpi = """
SELECT 
    a.nom_agence, 
    EXTRACT(MONTH FROM t.date_transaction) as mois,
    ROUND(AVG(t.montant_eur), 2) as moyenne_mensuelle
FROM transactions t
JOIN agences a ON t.id_agence = a.id_agence
GROUP BY a.nom_agence, mois
ORDER BY a.nom_agence, mois;
"""

result = pd.read_sql(query_kpi, engine)
print(result.head())

# Requête : Clients avec un solde (fictif ici basé sur montant) inférieur à la moyenne
query_sub = """
SELECT client_id, SUM(montant_eur) as solde
FROM transactions
GROUP BY client_id
HAVING SUM(montant_eur) < (SELECT AVG(montant_eur) FROM transactions)
LIMIT 10;
"""
print(pd.read_sql(query_sub, engine))